In [0]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 100000

df = pd.DataFrame({
    'age':                np.random.randint(20, 70, n),
    'income':             np.random.randint(20000, 200000, n),
    'loan_amount':        np.random.randint(5000, 500000, n),
    'debt_ratio':         np.round(np.random.uniform(0.1, 0.9, n), 2),
    'credit_score':       np.random.randint(300, 850, n),
    'num_late_payments':  np.random.randint(0, 20, n),
    'employment_years':   np.random.randint(0, 40, n),
    'monthly_expenses':   np.random.randint(1000, 20000, n),
    'savings_balance':    np.random.randint(0, 500000, n),
    'credit_utilization': np.round(np.random.uniform(0.0, 1.0, n), 2),
    'num_credit_lines':   np.random.randint(1, 15, n),
    'num_dependents':     np.random.randint(0, 5, n),
    'previous_default':   np.random.randint(0, 2, n),
})

df['default'] = (
    (df['debt_ratio'] > 0.6) &
    (df['credit_score'] < 500) &
    (df['num_late_payments'] > 5)
).astype(int)

# 故意注入髒資料
dirty_idx = np.random.choice(n, 5000, replace=False)
df.loc[dirty_idx[:1000], 'age'] = -1          # 不合理的年齡
df.loc[dirty_idx[1000:2000], 'credit_score'] = 9999  # 不合理的信用分數
df.loc[dirty_idx[2000:3000], 'income'] = None        # null 值
df.loc[dirty_idx[3000:4000], 'debt_ratio'] = -0.5    # 負數的 debt ratio
df.loc[dirty_idx[4000:5000], 'loan_amount'] = None   # null 值

spark_df = spark.createDataFrame(df)
spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_credit_raw")

print(f"完成：{spark_df.count()} 筆")
print(f"違約比例：{df['default'].mean():.2%}")